# EDA + modèle baseline sur le BAAC (2019-2023), sans enrichissement externe

Le CDC prévoit de trancher l'intégration trafic/bulletins après un modèle baseline (cf.
`CDC_GRAVIA.md` §13.6). Ce notebook fournit cette référence : uniquement les 4 tables BAAC
natives (caractéristiques, lieux, véhicules, usagers), sans météo/géo/trafic/bulletins. L'EDA
elle-même (`run_eda`, plus bas) est volontairement légère — distribution de la cible par année,
taux de valeurs nulles — pas une exploration visuelle : aucune bibliothèque de graphiques
n'est utilisée ici (voir `eda_raw_baac.ipynb` et `eda_exploration_baac.ipynb` pour l'EDA
visuelle, respectivement sur le brut et sur Gold).

**Périmètre** : 2019-2023 (schéma BAAC stable sur cette période — cf. pièges ci-dessous), toute
la France (pas de restriction géographique, contrairement à l'exploration trafic Paris qui
reste un enrichissement optionnel séparé).

**Pièges de schéma constatés** en comparant les 5 millésimes (le BAAC change de convention
chaque année, ce n'est pas anecdotique) :
- `Num_Acc` renommé `Accident_Id` dans `carcteristiques-2022.csv` uniquement.
- Fichier caractéristiques 2021/2022 orthographié « carcteristiques » (sans le « a ») par le
  producteur — pas une coquille de notre part, le nom réel du fichier sur data.gouv.fr.
- `jour`/`mois` : zéro-paddés certaines années (`"05"`), pas d'autres (`"5"`) → toujours caster
  en `Int64`, ne jamais dépendre du format texte.
- `id_usager` absent des usagers 2019-2020 (ajouté à partir de 2021) → non utilisé ici de toute
  façon (jointure sur `Num_Acc` uniquement).

**Anti-leakage** (cf. `CLAUDE.md`) : on exclut explicitement `secu1/2/3` (équipement de
sécurité, connu seulement après enquête) et `manv` (manœuvre détaillée, reconstituée après
enquête). Le `lat`/`long` brut n'est pas utilisé comme feature (donnée personnelle à
agréger/pseudonymiser plus tard, cf. `Gouvernance_GRAVIA.md`) ; seul le département (`dep`)
sert de granularité géographique pour ce baseline.

**Simplification assumée** : les caractéristiques véhicule (`catv`, `choc`, `obs`, `obsm`...)
ne sont pas encore agrégées au niveau accident dans ce premier baseline — seul le nombre de
véhicules impliqués est retenu. Le détail véhicule est un candidat pour une itération
ultérieure de feature engineering (`ml/features/`), pas pour cette référence.

In [1]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import polars as pl
from sklearn.metrics import classification_report, f1_score, recall_score

TARGET_RECALL = 0.80  # seuil CDC, priorité au recall (coût asymétrique : rater un grave coûte
# plus cher qu'une fausse alerte, cf. Presentation_Projet_GRAVIA_Jedha.md).

# Exécuté avec le répertoire de travail à la racine du dépôt.
BAAC_DIR = Path("data/raw/baac")

# (année -> nom de fichier caractéristiques, coquilles/renommages inclus)
CARACT_FILES = {
    2019: "caracteristiques-2019.csv",
    2020: "caracteristiques-2020.csv",
    2021: "carcteristiques-2021.csv",
    2022: "carcteristiques-2022.csv",
    2023: "caract-2023.csv",
}

# Features anti-leakage retenues pour ce baseline (connues au moment du signalement).
CARACT_FEATURES = ["lum", "dep", "agg", "int", "atm", "col"]
LIEUX_FEATURES = ["catr", "circ", "nbv", "vosp", "prof", "plan", "surf", "infra", "situ", "vma"]
CATEGORICAL_FEATURES = [
    "lum",
    "dep",
    "agg",
    "int",
    "atm",
    "col",
    "catr",
    "circ",
    "vosp",
    "prof",
    "plan",
    "surf",
    "infra",
    "situ",
]
NUMERIC_FEATURES = ["nbv", "vma", "heure", "nb_vehicules", "mois", "jour_semaine"]

## Chargement des 5 millésimes

In [2]:
def load_caracteristiques(year: int) -> pl.DataFrame:
    df = pl.read_csv(BAAC_DIR / CARACT_FILES[year], separator=";", infer_schema_length=0)
    id_col = "Accident_Id" if "Accident_Id" in df.columns else "Num_Acc"
    return df.select(
        pl.col(id_col).alias("num_acc"),
        pl.lit(year).alias("annee"),
        pl.col("mois").cast(pl.Int64, strict=False),
        pl.col("jour").cast(pl.Int64, strict=False),
        pl.col("hrmn").str.slice(0, 2).cast(pl.Int64, strict=False).alias("heure"),
        *[pl.col(c) for c in CARACT_FEATURES],
    )


def load_lieux(year: int) -> pl.DataFrame:
    df = pl.read_csv(BAAC_DIR / f"lieux-{year}.csv", separator=";", infer_schema_length=0)
    return df.select(
        pl.col("Num_Acc").alias("num_acc"), *[pl.col(c) for c in LIEUX_FEATURES]
    ).unique(subset=["num_acc"], keep="first")  # un accident peut avoir plusieurs lignes


def load_vehicules_count(year: int) -> pl.DataFrame:
    df = pl.read_csv(BAAC_DIR / f"vehicules-{year}.csv", separator=";", infer_schema_length=0)
    return df.group_by(pl.col("Num_Acc").alias("num_acc")).agg(
        pl.col("num_veh").n_unique().alias("nb_vehicules")
    )


def load_label(year: int) -> pl.DataFrame:
    df = pl.read_csv(BAAC_DIR / f"usagers-{year}.csv", separator=";", infer_schema_length=0)
    return df.group_by(pl.col("Num_Acc").alias("num_acc")).agg(
        pl.col("grav").cast(pl.Int64, strict=False).is_in([2, 3]).any().alias("is_grave")
    )


def load_year(year: int) -> pl.DataFrame:
    caract = load_caracteristiques(year)
    lieux = load_lieux(year)
    vehicules = load_vehicules_count(year)
    label = load_label(year)

    return (
        caract.join(lieux, on="num_acc", how="left")
        .join(vehicules, on="num_acc", how="left")
        .join(label, on="num_acc", how="left")
        .with_columns(
            pl.date(pl.col("annee"), pl.col("mois"), pl.col("jour"))
            .dt.weekday()
            .alias("jour_semaine")
        )
    )

In [3]:
print("Chargement 2019-2023...")
years = [load_year(y) for y in range(2019, 2024)]
full = pl.concat(years, how="diagonal_relaxed").drop_nulls("is_grave")
print(f"{full.height} accidents chargés.")

Chargement 2019-2023...


273226 accidents chargés.


## EDA — distribution de la cible et valeurs manquantes

In [4]:
def run_eda(df: pl.DataFrame) -> None:
    print("Accidents totaux (2019-2023) :", df.height)
    print()
    print("Distribution de la cible par année :")
    print(
        df.group_by("annee")
        .agg(pl.len().alias("n"), pl.col("is_grave").mean().alias("taux_grave"))
        .sort("annee")
    )
    print()
    print("Taux de valeurs manquantes (NULL SQL) par feature :")
    n = df.height
    cols = CARACT_FEATURES + LIEUX_FEATURES + ["nb_vehicules"]
    missing = df.select([(pl.col(c).is_null().sum() / n).alias(c) for c in cols])
    for c in missing.columns:
        print(f"  {c:15s} {missing[c][0]:.1%}")


run_eda(full)

Accidents totaux (2019-2023) : 273226

Distribution de la cible par année :
shape: (5, 3)
┌───────┬───────┬────────────┐
│ annee ┆ n     ┆ taux_grave │
│ ---   ┆ ---   ┆ ---        │
│ i32   ┆ u32   ┆ f64        │
╞═══════╪═══════╪════════════╡
│ 2019  ┆ 58840 ┆ 0.360656   │
│ 2020  ┆ 47744 ┆ 0.360904   │
│ 2021  ┆ 56518 ┆ 0.345961   │
│ 2022  ┆ 55302 ┆ 0.360041   │
│ 2023  ┆ 54822 ┆ 0.360895   │
└───────┴───────┴────────────┘

Taux de valeurs manquantes (NULL SQL) par feature :
  lum             0.0%
  dep             0.0%
  agg             0.0%
  int             0.0%
  atm             0.0%
  col             0.0%
  catr            0.0%
  circ            0.0%
  nbv             0.0%
  vosp            0.0%
  prof            0.0%
  plan            0.0%
  surf            0.0%
  infra           0.0%
  situ            0.0%
  vma             0.0%
  nb_vehicules    0.0%


Taux de gravité stable autour de 35-36 % chaque année. Bien plus élevé que le ~9 % observé sur
le sous-ensemble Paris (cf. exploration trafic) : cohérent, le mix national inclut les
accidents ruraux/autoroutiers à vitesse élevée, plus sévères que la moyenne urbaine dense
parisienne.

0 % de valeurs NULL SQL sur les features retenues (contraste net avec les sources trafic
externes, où le bruit/manquant était le problème central) — **mais ce chiffre ne compte pas la
sentinelle BAAC `" -1"`** (non renseigné, cf. `CLAUDE.md`, pièges de schéma) puisque les
colonnes sont lues en texte sans nettoyage ici : le non-renseigné explicite atteint jusqu'à
5,9 % sur `circ` (cf. `eda_raw_baac.ipynb`, qui le compte correctement). Écart assumé : ce
baseline n'a pas besoin de distinguer « non renseigné » de « absent » pour ses résultats,
contrairement à `ml/features` et Great Expectations qui, eux, la traitent explicitement.

## Split temporel et entraînement du baseline LightGBM

Train 2019-2021 / validation 2022 (calibrage du seuil de décision) / test 2023 (holdout jamais
vu, y compris pour le calibrage). Calibrer le seuil sur le test aurait été de la fuite de
méthodologie (le seuil ferait indûment mieux sur les données qu'il a servi à choisir).

In [5]:
def prepare_features(df: pl.DataFrame) -> pl.DataFrame:
    """Cast les catégorielles en category polars (LightGBM les gère nativement)."""
    return df.with_columns(
        [pl.col(c).cast(pl.Categorical) for c in CATEGORICAL_FEATURES]
    ).with_columns([pl.col(c).cast(pl.Float64, strict=False) for c in NUMERIC_FEATURES])


feature_cols = CARACT_FEATURES + LIEUX_FEATURES + ["nb_vehicules", "heure", "jour_semaine", "mois"]
feature_cols = list(dict.fromkeys(feature_cols))  # dédoublonne en gardant l'ordre
full = prepare_features(full)

train = full.filter(pl.col("annee") < 2022)
valid = full.filter(pl.col("annee") == 2022)
test = full.filter(pl.col("annee") == 2023)

print(f"Train      : {train.height} accidents ({train['is_grave'].mean():.1%} graves)")
print(f"Validation : {valid.height} accidents ({valid['is_grave'].mean():.1%} graves)")
print(f"Test       : {test.height} accidents ({test['is_grave'].mean():.1%} graves)")

Train      : 163102 accidents (35.6% graves)
Validation : 55302 accidents (36.0% graves)
Test       : 54822 accidents (36.1% graves)


In [6]:
def to_pandas_features(subset: pl.DataFrame):
    frame = subset.select(feature_cols).to_pandas()
    for c in CATEGORICAL_FEATURES:
        frame[c] = frame[c].astype("category")
    return frame


x_train, y_train = to_pandas_features(train), train["is_grave"].to_pandas().astype(int)
x_valid, y_valid = to_pandas_features(valid), valid["is_grave"].to_pandas().astype(int)
x_test, y_test = to_pandas_features(test), test["is_grave"].to_pandas().astype(int)

model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    class_weight="balanced",  # classe grave minoritaire (~10% à Paris, ~36% national)
    random_state=42,
    verbosity=-1,
)
model.fit(x_train, y_train, categorical_feature=CATEGORICAL_FEATURES)
print("Modèle entraîné.")

Modèle entraîné.


## Évaluation : seuil par défaut vs. seuil calibré sur la validation

In [7]:
# Seuil par défaut (0.5) à titre de comparaison.
proba_test = model.predict_proba(x_test)[:, 1]
y_pred_default = (proba_test >= 0.5).astype(int)

# Calibrage du seuil sur la VALIDATION (2022) pour viser le recall cible du CDC.
proba_valid = model.predict_proba(x_valid)[:, 1]
thresholds = np.linspace(0.01, 0.99, 99)
valid_recalls = [recall_score(y_valid, (proba_valid >= t).astype(int)) for t in thresholds]
eligible = [t for t, r in zip(thresholds, valid_recalls, strict=True) if r >= TARGET_RECALL]
tuned_threshold = max(eligible) if eligible else thresholds[int(np.argmax(valid_recalls))]

y_pred_tuned = (proba_test >= tuned_threshold).astype(int)

print(
    f"Seuil par défaut (0.5)   : recall={recall_score(y_test, y_pred_default):.3f}, "
    f"F1 macro={f1_score(y_test, y_pred_default, average='macro'):.3f}"
)
print(f"Seuil calibré sur 2022 ({tuned_threshold:.2f}) appliqué à 2023 (holdout) :")
recall_grave = recall_score(y_test, y_pred_tuned)
f1_macro = f1_score(y_test, y_pred_tuned, average="macro")
print(f"  Recall classe grave : {recall_grave:.3f}  (seuil CDC >= 0.80)")
print(f"  F1-score macro       : {f1_macro:.3f}  (seuil CDC >= 0.70)")
print()
print(classification_report(y_test, y_pred_tuned, target_names=["non_grave", "grave"]))

Seuil par défaut (0.5)   : recall=0.760, F1 macro=0.721
Seuil calibré sur 2022 (0.44) appliqué à 2023 (holdout) :
  Recall classe grave : 0.808  (seuil CDC >= 0.80)
  F1-score macro       : 0.708  (seuil CDC >= 0.70)

              precision    recall  f1-score   support

   non_grave       0.86      0.66      0.75     35037
       grave       0.57      0.81      0.67     19785

    accuracy                           0.71     54822
   macro avg       0.72      0.73      0.71     54822
weighted avg       0.76      0.71      0.72     54822



**Les deux seuils CDC sont atteints** (recall ≥ 0,80, F1 macro ≥ 0,70), avec une méthodologie
propre (aucune fuite : le seuil n'est jamais choisi sur les données d'évaluation finale). →
Référence baseline posée sans aucun enrichissement externe.

## Importance des features

In [8]:
importances = (
    pl.DataFrame({"feature": feature_cols, "importance": model.feature_importances_})
    .sort("importance", descending=True)
    .head(10)
)
print(importances)

shape: (10, 2)
┌──────────────┬────────────┐
│ feature      ┆ importance │
│ ---          ┆ ---        │
│ str          ┆ i32        │
╞══════════════╪════════════╡
│ dep          ┆ 3317       │
│ heure        ┆ 790        │
│ vma          ┆ 670        │
│ col          ┆ 555        │
│ nbv          ┆ 522        │
│ mois         ┆ 485        │
│ catr         ┆ 377        │
│ jour_semaine ┆ 342        │
│ int          ┆ 258        │
│ infra        ┆ 251        │
└──────────────┴────────────┘


`dep` (département) est de loin la feature la plus importante — cohérent avec l'hétérogénéité
géographique de la sévérité (urbain/rural, vitesses autorisées différentes par zone). Le
trafic Paris (signal confirmé, cf. `explo_trafic_paris_correlation_annuel.ipynb`) est un
candidat pour améliorer ce baseline sur le sous-ensemble parisien ; reste à mesurer le gain
réel par rapport à cette référence avant de l'intégrer en production (cf.
`eval_trafic_gain_paris.ipynb`).